Install notebook dependencies with `python -m pip install -e ".[notebooks]"`.

Training and validation use the same scheduled teacher-forcing ratio. Validation disables dropout and gradients; training loss is collected while weights change. Older runs used zero teacher forcing for validation, so compare only matching validation policies. Historical runs may use batch-averaged losses; new runs use non-padding-token means. Compare runs only when their metric definitions and splits match.

# Training dashboard

Explore train/validation loss, the validation − training difference, teacher forcing, and translation metrics. The notebook reads either a `*.metrics.json` file written by `scripts/train.py` or a resumable `*.latest.pth` checkpoint. Run all cells after training, or set `ARTIFACT_PATH` below to compare a particular run.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
ARTIFACT_PATH = None  # Example: ROOT / "artifacts/checkpoints/model-v1.metrics.json"
PLOT_TEMPLATE = "plotly_white"
TF_COLORSCALE = "Viridis"
pd.set_option("display.precision", 4)

## Load training history

Metric JSON files are preferred because they are small. If none exists, the newest resumable checkpoint is used. Set `ARTIFACT_PATH` explicitly if several experiments are present.

In [ ]:
def discover_training_artifacts(root):
    patterns = ("artifacts/checkpoints/*.metrics.json", "artifacts/checkpoints/*.latest.pth")
    return sorted(
        (path for pattern in patterns for path in root.glob(pattern)),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

def load_history(path):
    path = Path(path)
    if path.suffix == ".json":
        payload = json.loads(path.read_text(encoding="utf-8"))
        rows = payload if isinstance(payload, list) else payload.get("history", [])
    elif path.suffix in {".pth", ".pt"}:
        import torch
        payload = torch.load(path, map_location="cpu", weights_only=True)
        state = payload.get("training_state") or {}
        rows = state.get("history", [])
    else:
        raise ValueError(f"Unsupported artifact: {path}")
    required = {"epoch", "train_loss", "validation_loss", "teacher_forcing_ratio"}
    history = pd.DataFrame(rows)
    missing = required.difference(history.columns)
    if history.empty or missing:
        raise ValueError(f"No complete training history in {path}; missing: {sorted(missing)}")
    history = history.sort_values("epoch").reset_index(drop=True)
    history["validation_train_difference"] = history.validation_loss - history.train_loss
    return history

candidates = discover_training_artifacts(ROOT)
artifact_path = Path(ARTIFACT_PATH) if ARTIFACT_PATH else (candidates[0] if candidates else None)
if artifact_path is None:
    raise FileNotFoundError("No training history found. Run scripts/train.py or set ARTIFACT_PATH.")
history = load_history(artifact_path)
print(f"Loaded {len(history)} intervals from {artifact_path}")
history

## Run summary

In [ ]:
best_index = history.validation_loss.idxmin()
best = history.loc[best_index]
summary = pd.Series({
    "completed intervals": len(history),
    "best interval": int(best.epoch),
    "best validation loss": best.validation_loss,
    "train loss at best": best.train_loss,
    "final train loss": history.iloc[-1].train_loss,
    "final validation loss": history.iloc[-1].validation_loss,
    "final validation − training difference": history.iloc[-1].validation_train_difference,
    "final teacher forcing": history.iloc[-1].teacher_forcing_ratio,
}, name="value")
summary.to_frame()

## Learning curves

In [ ]:
loss_figure = make_subplots(rows=1, cols=2, subplot_titles=("Loss by training interval", "Validation − training loss"))
marker_style = dict(
    color=history.teacher_forcing_ratio, colorscale=TF_COLORSCALE, size=10,
    cmin=0, cmax=1, colorbar=dict(
        title=dict(text="Teacher-forcing ratio", side="top"),
        orientation="h", x=0.5, xanchor="center", y=-0.24, yanchor="top", len=0.55, thickness=14,
    ),
)
hover = ("Interval %{x}<br>Loss %{y:.4f}<br>Teacher forcing %{marker.color:.3f}<extra>%{fullData.name}</extra>")
loss_figure.add_trace(go.Scatter(
    x=history.epoch, y=history.train_loss, name="Train loss", mode="lines+markers",
    line=dict(color="#2563eb"), marker=marker_style, hovertemplate=hover,
), row=1, col=1)
loss_figure.add_trace(go.Scatter(
    x=history.epoch, y=history.validation_loss, name="Validation loss", mode="lines+markers",
    line=dict(color="#f97316"), marker={**marker_style, "showscale": False}, hovertemplate=hover,
), row=1, col=1)
loss_figure.add_trace(go.Scatter(
    x=[best.epoch], y=[best.validation_loss], name="Best validation", mode="markers",
    marker=dict(symbol="star", size=17, color="#16a34a"),
    hovertemplate="Best interval %{x}<br>Validation loss %{y:.4f}<extra></extra>",
), row=1, col=1)
loss_figure.add_trace(go.Scatter(
    x=history.epoch, y=history.validation_train_difference, name="Validation − training loss", mode="lines+markers",
    line=dict(color="#dc2626"), marker={**marker_style, "showscale": False},
    hovertemplate=hover.replace("Loss", "Gap"),
), row=1, col=2)
loss_figure.add_hline(y=0, line_width=1, line_color="#64748b", row=1, col=2)
loss_figure.update_xaxes(title_text="Interval")
loss_figure.update_yaxes(title_text="Cross-entropy loss", row=1, col=1)
loss_figure.update_yaxes(title_text="Validation loss − train loss", row=1, col=2)
loss_figure.update_layout(
    template=PLOT_TEMPLATE, title=artifact_path.name, height=610, hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="left", x=0),
    margin=dict(t=115, b=130, l=70, r=45),
)
loss_figure.show()

Training and validation share the teacher-forcing ratio, but dropout and weight updates still differ. Interpret the loss difference alongside BLEU/chrF; changing the teacher-forcing ratio also changes the validation task between intervals.

In [ ]:
tf_figure = go.Figure(go.Scatter(
    x=history.epoch, y=history.teacher_forcing_ratio, mode="lines+markers",
    line=dict(color="#7c3aed", width=3),
    marker=dict(
        color=history.teacher_forcing_ratio, colorscale=TF_COLORSCALE,
        cmin=0, cmax=1, size=12, showscale=True, colorbar=dict(title="Ratio"),
    ),
    fill="tozeroy", fillcolor="rgba(124, 58, 237, 0.10)",
    hovertemplate="Interval %{x}<br>Teacher forcing %{y:.3f}<extra></extra>",
))
tf_figure.update_layout(
    template=PLOT_TEMPLATE, title="Teacher-forcing schedule", height=430,
    xaxis_title="Interval", yaxis_title="Teacher-forcing ratio",
    yaxis_range=[-0.02, 1.02], hovermode="x",
)
tf_figure.show()

## BLEU and chrF evaluation metrics

This section discovers the repository's evaluation reports. BLEU and chrF are corpus-level translation-quality metrics; compare scores only when they use the same evaluation corpus and settings.

In [ ]:
def evaluation_rows(root):
    rows = []
    for path in sorted(root.glob("artifacts/evaluations/**/results.json")):
        payload = json.loads(path.read_text(encoding="utf-8"))
        for model_name, model in payload.get("models", {}).items():
            for dataset_name, scores in model.get("datasets", {}).items():
                if "bleu" in scores or "chrf" in scores:
                    rows.append({
                        "report": str(path.parent.relative_to(root)),
                        "model": model_name,
                        "dataset": dataset_name,
                        "bleu": scores.get("bleu"),
                        "chrf": scores.get("chrf"),
                    })
    return pd.DataFrame(rows)

evaluation = evaluation_rows(ROOT)
if evaluation.empty:
    print("No evaluation results found under artifacts/evaluations/.")
else:
    display(evaluation.sort_values(["dataset", "bleu"], ascending=[True, False]))

In [ ]:
if not evaluation.empty:
    metric_data = evaluation.assign(run=evaluation.model + " · " + evaluation.dataset).melt(
        id_vars=["report", "model", "dataset", "run"],
        value_vars=["bleu", "chrf"], var_name="metric", value_name="score",
    )
    metric_figure = px.bar(
        metric_data, x="score", y="run", color="metric", barmode="group", orientation="h",
        color_discrete_map={"bleu": "#2563eb", "chrf": "#f97316"},
        custom_data=["report", "dataset"], template=PLOT_TEMPLATE,
        title="Translation quality by model and dataset", labels={"run": "Run", "score": "Score", "metric": "Metric"},
    )
    metric_figure.update_traces(hovertemplate="%{y}<br>%{fullData.name}: %{x:.2f}<br>Report: %{customdata[0]}<extra></extra>")
    metric_figure.update_layout(height=max(440, len(evaluation) * 55), legend_title_text="Metric (higher is better)")
    metric_figure.show()

## Interpretation checklist

- Prefer the checkpoint from the lowest validation loss, not automatically the last interval.
- If both losses remain high, train longer or revisit capacity, optimization, and data quality.
- If train loss falls while validation loss worsens, consider earlier stopping or stronger regularization.
- Use BLEU/chrF and manual translation examples alongside loss; loss alone does not measure readable translations.